# Retail Customer Segmentation using K-Means Clustering
### End-to-End Machine Learning Tutorial in Python

Welcome to this beginner-friendly tutorial on **Customer Segmentation** using the classic **Mall Customers** dataset from Kaggle.

**Objective**:
Segment customers into actionable groups based on annual income and spending score (optionally age), allowing retail marketing teams to design tailored campaigns, maximize ROI, and increase customer lifetime value (LTV).

**Table of Contents**:
1. Environment Setup & Data Loading
2. Exploratory Data Analysis (EDA)
3. Data Preprocessing & Feature Scaling
4. Determining Optimal K (Elbow Method & Silhouette Score)
5. Model Training (Scikit-Learn vs. NumPy From-Scratch KMeans)
6. 2D & 3D Interactive Visualizations
7. Business Interpretation & Marketing Strategy Playbook
8. Customer Inference Function & Model Persistence

## 1. Setup & Data Loading

In [ ]:
# Install dependencies (if running in Google Colab)
# !pip install pandas numpy matplotlib seaborn scikit-learn plotly joblib

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
import plotly.express as px
import plotly.graph_objects as go
import joblib

sns.set_theme(style="whitegrid", palette="muted")
print("Libraries successfully imported!")

In [ ]:
# In Google Colab, you can upload Mall_Customers.csv using:
# from google.colab import files
# uploaded = files.upload()

DATASET_PATH = "Mall_Customers.csv"
df = pd.read_csv(DATASET_PATH)

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
df.head()

In [ ]:
# Inspect column data types and missing values
print("--- Dataframe Info ---")
df.info()

print("\n--- Missing Values ---")
print(df.isnull().sum())

print("\n--- Summary Statistics ---")
df.describe().round(2)

## 2. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Customer Demographic & Behavioral Distributions", fontsize=16, weight="bold")

# 1. Gender Pie Chart
gender_counts = df["Gender"].value_counts()
axes[0, 0].pie(gender_counts, labels=gender_counts.index, autopct="%1.1f%%", startangle=140, colors=["#3b82f6", "#ec4899"], explode=(0.04, 0))
axes[0, 0].set_title("Gender Breakdown", weight="bold")

# 2. Age Distribution
sns.histplot(df["Age"], kde=True, ax=axes[0, 1], color="#8b5cf6", bins=15)
axes[0, 1].axvline(df["Age"].mean(), color="crimson", linestyle="--", label=f"Mean: {df['Age'].mean():.1f}")
axes[0, 1].set_title("Age Distribution", weight="bold")
axes[0, 1].legend()

# 3. Income Distribution
sns.histplot(df["Annual Income (k$)"], kde=True, ax=axes[1, 0], color="#10b981", bins=15)
axes[1, 0].axvline(df["Annual Income (k$)"].mean(), color="crimson", linestyle="--", label=f"Mean: ${df['Annual Income (k$)'].mean():.1f}k")
axes[1, 0].set_title("Annual Income (k$)", weight="bold")
axes[1, 0].legend()

# 4. Spending Score Distribution
sns.histplot(df["Spending Score (1-100)"], kde=True, ax=axes[1, 1], color="#f59e0b", bins=15)
axes[1, 1].axvline(df["Spending Score (1-100)"].mean(), color="crimson", linestyle="--", label=f"Mean: {df['Spending Score (1-100)'].mean():.1f}")
axes[1, 1].set_title("Spending Score (1-100)", weight="bold")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 3. Data Preprocessing & Feature Scaling

K-Means computes Euclidean distance ($d = \sqrt{\sum (x_i - y_i)^2}$). Therefore, features with larger scales will artificially dominate the cluster centroids. We use `StandardScaler` to ensure mean = 0 and standard deviation = 1.

In [ ]:
# Drop CustomerID (arbitrary key with no behavioral signal)
features_2d = ["Annual Income (k$)", "Spending Score (1-100)"]
X_raw = df[features_2d].values

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f"Scaled Feature Matrix Shape: {X_scaled.shape}")
print("Sample scaled values:\n", np.round(X_scaled[:5], 3))

## 4. Finding the Optimal K (Elbow Method & Silhouette Score)

In [ ]:
wcss = []
sil_scores = []
k_range = range(1, 11)
k_sil = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    km.fit(X_scaled)
    wcss.append(km.inertia_)
    if k >= 2:
        sil_scores.append(silhouette_score(X_scaled, km.labels_))

# Plot Elbow and Silhouette
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].plot(k_range, wcss, marker="o", color="#3b82f6", lw=2)
ax[0].axvline(5, color="crimson", linestyle="--", label="Elbow at K=5")
ax[0].set_title("Elbow Method (WCSS vs K)", weight="bold")
ax[0].set_xlabel("Number of Clusters (K)")
ax[0].set_ylabel("Inertia (WCSS)")
ax[0].legend()

ax[1].plot(k_sil, sil_scores, marker="s", color="#10b981", lw=2)
ax[1].axvline(5, color="crimson", linestyle="--", label="Optimal K=5")
ax[1].set_title("Silhouette Score vs K", weight="bold")
ax[1].set_xlabel("Number of Clusters (K)")
ax[1].set_ylabel("Silhouette Coefficient")
ax[1].legend()

plt.tight_layout()
plt.show()

print("JUSTIFICATION FOR K=5:")
print("1. The Elbow curve clearly exhibits an inflection bend at K=5.")
print("2. The Silhouette coefficient is high (~0.55), indicating well-separated clusters.")
print("3. 5 clusters map cleanly into practical retail archetypes (Low/High Income x Low/High Spending + Average).")

## 5. Model Training: Scikit-Learn vs. NumPy From-Scratch KMeans

In [ ]:
class KMeansScratch:
    """Custom NumPy implementation of K-Means clustering."""
    def __init__(self, n_clusters=5, max_iter=300, tol=1e-4, random_state=42):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state
        
    def fit(self, X):
        rng = np.random.default_rng(self.random_state)
        n_samples, n_features = X.shape
        
        # 1. K-Means++ Initialization
        centroids = np.empty((self.n_clusters, n_features))
        centroids[0] = X[rng.choice(n_samples)]
        for c in range(1, self.n_clusters):
            dists = np.min(np.sum((X[:, np.newaxis, :] - centroids[:c]) ** 2, axis=2), axis=1)
            probs = dists / np.sum(dists)
            centroids[c] = X[rng.choice(n_samples, p=probs)]
            
        # EM Iteration Loop
        for it in range(self.max_iter):
            # Assign samples to nearest centroid
            distances = np.linalg.norm(X[:, np.newaxis, :] - centroids[np.newaxis, :, :], axis=2)
            labels = np.argmin(distances, axis=1)
            
            # Recompute centroids
            new_centroids = np.array([X[labels == k].mean(axis=0) for k in range(self.n_clusters)])
            shift = np.max(np.linalg.norm(new_centroids - centroids, axis=1))
            centroids = new_centroids
            if shift < self.tol:
                break
                
        self.cluster_centers_ = centroids
        self.labels_ = labels
        self.inertia_ = np.sum((X - centroids[labels]) ** 2)
        return self

# 1. Train Scikit-Learn KMeans
sk_model = KMeans(n_clusters=5, init="k-means++", n_init=10, random_state=42)
sk_labels = sk_model.fit_predict(X_scaled)

# 2. Train Scratch NumPy KMeans
scratch_model = KMeansScratch(n_clusters=5, random_state=42).fit(X_scaled)

print(f"Scikit-Learn Inertia : {sk_model.inertia_:.4f}")
print(f"NumPy Scratch Inertia: {scratch_model.inertia_:.4f}")
print(f"Cluster Agreement (Adjusted Rand Index): {adjusted_rand_score(sk_labels, scratch_model.labels_) * 100:.2f}%")

## 6. Visualizations & Customer Archetypes

In [ ]:
# Add cluster labels to dataframe
df["Cluster"] = sk_labels

# Map clusters to business archetypes based on centers
unscaled_centers = scaler.inverse_transform(sk_model.cluster_centers_)

plt.figure(figsize=(12, 7))
palette = ["#3b82f6", "#10b981", "#8b5cf6", "#f59e0b", "#ef4444"]
sns.scatterplot(
    data=df,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    hue="Cluster",
    palette=palette,
    s=85,
    alpha=0.9,
    edgecolor="black"
)

# Plot Centroids
plt.scatter(
    unscaled_centers[:, 0],
    unscaled_centers[:, 1],
    s=300,
    c="white",
    marker="X",
    edgecolors="black",
    linewidths=2,
    label="Centroids"
)

plt.title("Customer Segmentation (Income vs Spending Score)", weight="bold", fontsize=14)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Per-Cluster Feature Averages
summary = df.groupby("Cluster").agg(
    Count=("Age", "count"),
    Avg_Age=("Age", "mean"),
    Avg_Income=("Annual Income (k$)", "mean"),
    Avg_Spending_Score=("Spending Score (1-100)", "mean")
).round(2)
summary["Pct_Share"] = (summary["Count"] / len(df) * 100).round(1).astype(str) + "%"
summary

## 7. Business Interpretation & Marketing Strategy Playbook

| Segment | Persona Name | Behavior | Targeted Marketing Action |
|---|---|---|---|
| **0** | **Target VIPs** | High Income, High Spending | White-glove concierge, private previews, luxury line early access, bespoke rewards |
| **1** | **Careful Spenders** | High Income, Low Spending | High-ROI durability messaging, premium warranties, investment-grade items |
| **2** | **Steady Middle** | Average Income, Average Spending | Seasonal promotions, bundle discounts, tiered loyalty points |
| **3** | **Trend Seekers** | Low Income, High Spending | Buy Now Pay Later (BNPL), student discounts, viral product drops, flash promotions |
| **4** | **Budget Conscious** | Low Income, Low Spending | Clearance notifications, value multipacks, price match guarantees |

## 8. Inference Function & Model Persistence

In [ ]:
def predict_segment(income: float, score: float, scaler, model):
    """Predict cluster for a new customer based on Annual Income ($k) and Spending Score."""
    raw_point = np.array([[income, score]])
    scaled_point = scaler.transform(raw_point)
    pred = int(model.predict(scaled_point)[0])
    return pred

# Test inference
new_income = 88.0
new_score = 92.0
pred_c = predict_segment(new_income, new_score, scaler, sk_model)
print(f"New Customer ($88k income, 92 spending score) -> Cluster {pred_c}")

# Save Model Artifacts
joblib.dump(sk_model, "kmeans_model.joblib")
joblib.dump(scaler, "scaler.joblib")
df.to_csv("customer_segments.csv", index=False)
print("Artifacts successfully saved to disk:")
print("- customer_segments.csv")
print("- kmeans_model.joblib")
print("- scaler.joblib")